# Introdução

Digamos que eu esteja me sentindo de uma maneira muito específica, e eu queira ouvir uma música que consoe meu estado de espírito. Para isso, podemos usar técnicas de Processamento de Linguagem Natural (PLN) para analisar o sentimento de uma frase ou texto e, em seguida, recomendar uma música que corresponda a esse sentimento.

Lógicamente, o universo musical é vasto e diverso, há todavia um artista com tamanho alcance e talento que faz com que todos os outros sejam irrelevantes: David Bowie.

Dessa forma, podemos criar um chatbot que, dada uma descrição do seu estado de espírito, responda a música ideal para enriquecer o seu ser.

# Dados

## Obtenção

Em primeiro lugar é necessário obter todas as letras das músicas de David Bowie. Para tanto, foi utilizado o site [Bowie Wonderworld](https://www.bowiewonderworld.com/songs/dblyrics.htm), que contém todas as letras das músicas do artista. A partir desse site, é possível realizar uma cópia bruta (é necessário desativar os scripts do site para tanto) de todas as letras para um arquivo de texto.

## Transformação e anotação

Primeiramente é necessário transformar o arquivo bruto em um arquivo estruturado, de forma que cada música seja representada por uma linha, contendo o título da música, a letra e o sentimento.

Devido ao altíssimo volume de músicas, a anotação manual de sentimentos para cada música seria inviável.
Por tanto, conclui-se que a a utilização de assistência de IA seria a solução mais apropriada.

A transformação e anotação ocorreram por tanto através do Copilot,
 utilizado o modelo GPT-5.6 Sol, janela de contexto de e raciocínio padrão com o seguinte prompt:


```
I need you to create a csv with the title of each song, the lyrics of each song, and a sentence describing the feelings of the song
```


## Aprimoração

O resultado das anotações iniciais todavia não era satisfatório, sentimentos genéricos e muitas vezes repetitivos foram atribuídos a músicas com sentimentos distintos.

As estratégias adotadas para aprimorar as anotações foram as seguintes:

1. Configurar a janela de contexto para 1.1M tokens
2. Aumentar o nível de raciocínio para "xhigh"
3. Descrever melhor o prompt, incluindo exemplos de sentimentos para músicas específicas. O prompt final utilizado foi o seguinte:

```
I need the feeling field to be more descriptive and individual to each song, for instance, Word on a Wing is a song that evokes a feeling of resignation to a higher power, a plea with God for direction, you can look the net for explanations too
```


# Pré-processamento

## Inicializando o ambiente

In [1]:
import pandas as pd
import numpy as np

## Importando dados

In [2]:
try:
    lyrics = pd.read_csv("bowie_songs.csv")
except FileNotFoundError:
    lyrics = pd.read_csv("https://raw.githubusercontent.com/fabio-osti/maua/refs/heads/master/artificial%20intelligence/pln/atividades/bowie_songs.csv")

lyrics

,title,lyrics,feelings
0,1917,(Instrumental),"Wordless and murky, this piece leans on its Fi..."
1,1984,"Someday they won't let you, now you must agree...","A funk-driven warning shot, it wraps claustrop..."
2,1984/Dodo,"Someday they won't let you, but now you must a...","Stitching dystopian alarm to hushed gossip, th..."
3,5:15 The Angels Have Gone,"5:15\nI'm changing trains, this little town\nL...","Set on a rainy platform, this is a farewell we..."
4,'87 And Cry,It's just a one dollar secret\nA lover's secre...,"Frustration curdles into bitterness, a snarlin..."
...,...,...,...
591,Word On A Wing,"In this age of grand delusion, you walked into...","Performed live, the hymn sounds even more expo..."
592,Yassassin,CHORUS\n Yassassin - I'm not a moody guy\n Y...,"A migrant's weary plea for peace, pride and ex..."
593,You Better Tell Her,NaN,"Impatient counsel drives the phrase, somebody ..."
594,You Can't Sit Down,Hey pretty baby! (you can't sit down)\nA don't...,"Irresistible compulsion to move, the beat trea..."


## Limpando dados

In [3]:
lyrics_clean = lyrics.dropna()
lyrics_clean = lyrics_clean[~lyrics_clean['title'].str.contains("(demo)", regex=False)]
lyrics_clean

,title,lyrics,feelings
0,1917,(Instrumental),"Wordless and murky, this piece leans on its Fi..."
1,1984,"Someday they won't let you, now you must agree...","A funk-driven warning shot, it wraps claustrop..."
2,1984/Dodo,"Someday they won't let you, but now you must a...","Stitching dystopian alarm to hushed gossip, th..."
3,5:15 The Angels Have Gone,"5:15\nI'm changing trains, this little town\nL...","Set on a rainy platform, this is a farewell we..."
4,'87 And Cry,It's just a one dollar secret\nA lover's secre...,"Frustration curdles into bitterness, a snarlin..."
...,...,...,...
590,Without You I'm Nothing,Strange infatuation seems to grace the evening...,"Sultry self-abasement, decadent images sliding..."
591,Word On A Wing,"In this age of grand delusion, you walked into...","Performed live, the hymn sounds even more expo..."
592,Yassassin,CHORUS\n Yassassin - I'm not a moody guy\n Y...,"A migrant's weary plea for peace, pride and ex..."
594,You Can't Sit Down,Hey pretty baby! (you can't sit down)\nA don't...,"Irresistible compulsion to move, the beat trea..."


# Testes

Embora a tarefa seja subjetiva, é necessário, ainda assim, ao menos um teste que demonstre o comportamento do modelo.

Para tanto, foram criadas algumas sentenças descrevendo sentimentos específicos, e a música esperada para cada sentimento.

In [4]:
_test = [
    # Praticamente uma cópia do sentimento da música, serve como teste de sanidade
    ("I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.",
     "Station To Station"),
    # Esse pode ser acertado tanto pela letra quanto pelo sentimento
    ("I feel like I'm cracking under pressure and that only love can save me.", "Under Pressure"),
    # Relativamente fácil também, mas dá para errar
    ("I feel euphoric and want to dance with my beloved.", "Let's Dance"),
    # Começa a dificultar, as palavras não estão diretamente nem na letra nem no sentimento
    ("I'm feeling regretful and ashamed for falling so low on my addiction.", "Ashes To Ashes"),
    ("I feel resignation, I just want to understand God's plan for me.", "Word On A Wing"),
    # Esse é o mais difícil, é necessário interpretar o contexto
    ("I'm completely head over heels in love and want to deliver myself completely", "I Would Be Your Slave"),
    # Extremamente específica
    ("I'm in love with a chinese woman", "China Girl"),
    # Bonus, não tem resposta certa, mas é interessante ver o que o modelo sugere
    ("I'm feeling anxious and stressed about an upcoming event.", "???"),
]

def _ranking(indices: tuple[list[int], list[float]]):
    return '\n'.join(
        f"\t\t{i}. {lyrics_clean.iloc[song_idx]['title']}: {sim:.2%}"
        for i, (song_idx, sim) in enumerate(zip(indices[0][:3], indices[1][:3]), start=1)
    )

def _position(expected, feeling_rank, lyrics_rank):
    if expected not in lyrics_clean['title'].values:
        return -1, -1
    i = np.where(lyrics_clean['title'] == expected)[0][0]
    return (
        np.where(feeling_rank[0] == i)[0][0] + 1,
        np.where(lyrics_rank[0] == i)[0][0] + 1,
    )

def test_similarity_function(feeling_similarity_function, lyrics_similarity_function):
    f_rank = 0
    l_rank = 0
    c = 0
    for phrase, expected in _test:
        most_similar_by_feeling = feeling_similarity_function(phrase)
        most_similar_by_lyrics = lyrics_similarity_function(phrase)
        print(f"Input phrase: {phrase}")
        if expected != "???":
            (feeling_pos, lyrics_pos) = _position(expected, most_similar_by_feeling, most_similar_by_lyrics)
            print(f"\t* Expected song ({expected}) true position:\n\t\t* By feeling: {feeling_pos}º\n\t\t* By lyrics: {lyrics_pos}º")
            f_rank += feeling_pos
            l_rank += lyrics_pos
            c += 1
        print(f"\t* Most similar songs by feeling: \n{_ranking(most_similar_by_feeling)}")
        print(f"\t* Most similar songs by lyrics: \n{_ranking(most_similar_by_lyrics)}")
        print("\n")
    print(f"Average rank by feeling: {f_rank / c}")
    print(f"Average rank by lyrics: {l_rank / c}")

# Rankeando por similaridade

A abordagem principal do projeto é simples, comparar a similaridade da sentença descrevendo o sentimento do usuário com a sentença descrevendo o sentimento de cada música e também com a própria letra da música.

Para tanto, é necessário vetorizar as sentenças, utilizaremos diferentes técnicas de vetorização para comparar os resultados e escolher a melhor abordagem. A comparação será feita utilizando a métrica de similaridade cosseno, que mede o ângulo entre dois vetores, sendo 1 para vetores idênticos e 0 para vetores ortogonais.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def similaridade(de, para):
    similaridades = cosine_similarity(de, para)
    argsoted = similaridades.argsort()[0][::-1]
    return similaridades.argsort()[0][::-1], similaridades[0][argsoted]

# Vetorização

Para realizar a vetorização das sentenças, utilizaremos diferentes técnicas de vetorização, como TF-IDF, Word2Vec, Doc2Vec e Transformers. Cada técnica possui suas próprias características e vantagens, e será interessante comparar os resultados obtidos com cada uma delas.

## TF-IDF

A primeira abordagem, que servirá de base de comparação para as demais, é a utilização de TF-IDF (Term Frequency-Inverse Document Frequency) para transformar os sentimentos das músicas. Essa técnica permite identificar a importância de cada palavra em relação ao conjunto de documentos (neste caso, os sentimentos das músicas).

### Treinando o modelo

Começamos treinando o modelo TF-IDF para os sentimentos e para as letras das músicas, utilizando a biblioteca `sklearn`. A função `TfidfVectorizer` é utilizada para transformar o texto em uma matriz de TF-IDF.

Como o modelo TF-IDF já da um peso menor a palavras comuns, não é necessário remover stopwords, mas é possível fazer isso caso seja desejado.

Além disso, o modelo TF-IDF detém um próprio tokenizador, dessa forma, não é necessário implementar um tokenizador para essa abordagem.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

feelings_tfidf_vectorizer = TfidfVectorizer()
feelings_tfidf_matrix = feelings_tfidf_vectorizer.fit_transform(lyrics_clean['feelings'])

lyrics_tfidf_vectorizer = TfidfVectorizer()
lyrics_tfidf_matrix = lyrics_tfidf_vectorizer.fit_transform(lyrics_clean['lyrics'])

### Função de Similaridade

Vetorizamos a sentença de entrada utilizando o mesmo vetor TF-IDF utilizado para os sentimentos das músicas e calculamos a similaridade cosseno entre a sentença vetorizada e a matriz de sentimentos das músicas. A função `most_similar_tfidf` retorna os índices das músicas mais similares à sentença de entrada.

In [7]:
def most_similar_tfidf(sentence, matrix, vectorizer):
    phrase_vec = vectorizer.transform([sentence])
    return similaridade(phrase_vec, matrix)

def most_similar_by_feeling_tfidf(sentence):
    return most_similar_tfidf(sentence, feelings_tfidf_matrix, feelings_tfidf_vectorizer)

def most_similar_by_lyrics_tfidf(sentence):
    return most_similar_tfidf(sentence, lyrics_tfidf_matrix, lyrics_tfidf_vectorizer)

### Teste

In [8]:
test_similarity_function(
    most_similar_by_feeling_tfidf,
    most_similar_by_lyrics_tfidf,
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected song (Station To Station) true position:
		* By feeling: 1º
		* By lyrics: 167º
	* Most similar songs by feeling: 
		1. Station To Station: 96.17%
		2. Goodbye Mr. Ed: 13.55%
		3. Soul Love: 13.07%
	* Most similar songs by lyrics: 
		1. I Am With Name: 14.19%
		2. The Pretty Things Are Going To Hell: 11.55%
		3. When The Wind Blows: 10.24%


Input phrase: I feel like I'm cracking under pressure and that only love can save me.
	* Expected song (Under Pressure) true position:
		* By feeling: 64º
		* By lyrics: 1º
	* Most similar songs by feeling: 
		1. Cat People (Putting Out Fire): 35.29%
		2. "Helden": 24.50%
		3. Girls: 19.59%
	* Most similar songs by lyrics: 
		1. Under Pressure: 22.04%
		2. Under The God: 18.39%
		3. Love Me Do: 14.70%


Input phrase: I feel euphoric and want to dance with my beloved.
	* Expected song (Let's D

Podemos observar que a função de similaridade baseada em TF-IDF apresenta resultados razoáveis quando as palavras da sentença de entrada estão presentes na letra ou no sentimento da música. Todavia, pequenas variações acabam deteriorando a performance do modelo, o que pode ser muito bem verificado no teste de China Girl, pois, da sentença do usuário, as palavras "chinese" e "woman" não estão presentes na letra da música, o que faz com que o modelo não consiga identificar a música correta.

## Embedding

Fazendo o uso de embeddings pré-treinados, podemos melhorar a performance do modelo, pois embeddings são representações vetoriais de palavras ou frases que capturam o significado semântico das palavras, permitindo que palavras com significados semelhantes tenham representações vetoriais próximas no espaço vetorial. Dessa forma, mesmo que as palavras da sentença de entrada não estejam presentes na letra ou no sentimento da música, o modelo ainda pode identificar músicas com sentimentos semelhantes.

### Tokenização

Como os métodos de embedding Word2Vec e Doc2Vec não contem tokenizador próprio, é necessário implementar um tokenizador.

In [9]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

def tokenize(text, remove_stopwords=True):
    tokens = word_tokenize(text.lower())
    if remove_stopwords:
        tokens = [t for t in tokens if t.isalpha() and t not in stopwords.words('english')]
    else:
        tokens = [t for t in tokens if t.isalpha()]
    return tokens

[nltk_data] Downloading package punkt to /home/fabio/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/fabio/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Word2Vec

Para o método Word2Vec, utilizaremos o modelo pré-treinado do Google News, que contém vetores de palavras treinados em um grande corpus de notícias. Esse modelo é capaz de capturar relações semânticas entre palavras, permitindo que palavras com significados semelhantes tenham representações vetoriais próximas no espaço vetorial.

In [10]:
import gensim.downloader as api

# Load pre-trained Word2Vec model
word2vec_model = api.load("word2vec-google-news-300")
word2vec_model["love"][::10] # Example of getting the vector for a word

array([ 0.10302734, -0.02624512, -0.03857422, -0.11181641,  0.18652344,
       -0.10986328,  0.24316406,  0.28515625, -0.31054688, -0.00958252,
       -0.19726562, -0.22167969,  0.05566406, -0.140625  , -0.09375   ,
        0.27734375, -0.07763672,  0.06005859, -0.30664062,  0.10644531,
       -0.0390625 , -0.10839844, -0.07128906, -0.24804688,  0.04736328,
        0.07470703, -0.09179688,  0.07763672,  0.16113281, -0.03198242],
      dtype=float32)

#### Vetorização

A vetorização de sentença é feita calculando a média dos vetores das palavras que compõem a sentença. Se uma palavra não estiver presente no vocabulário do modelo, ela é ignorada. Se nenhuma palavra da sentença estiver presente no vocabulário, um vetor de zeros é retornado.

Aqui, removeremos as stopwords, pois elas não contribuem para o significado da sentença e podem distorcer a média dos vetores das palavras.

In [11]:
from gensim.models import KeyedVectors

def avg_w2v_vec(text, w2v_model: KeyedVectors):
    tokens = tokenize(text.lower())
    vectors = [
        w2v_model[t]
        for t in tokens
        if t in w2v_model
    ]

    if not vectors:
        return np.zeros(w2v_model.vector_size)

    return np.mean(vectors, axis=0)

feelings_word2vec_matrix = [
    avg_w2v_vec(feeling, word2vec_model)
    for feeling in lyrics_clean['feelings']
]

lyrics_word2vec_matrix = [
    avg_w2v_vec(lyric, word2vec_model)
    for lyric in lyrics_clean['lyrics']
]

#### Função de Similaridade

A função de similaridade é implementada da mesma forma que a função de similaridade baseada em TF-IDF, mas utilizando a vetorização de sentença baseada em Word2Vec.

In [12]:
def most_similar_w2v(sentence, matrix):
    phrase_vec = avg_w2v_vec(sentence, word2vec_model).reshape(1, -1)
    return similaridade(phrase_vec, matrix)

def most_similar_by_feeling_w2v(sentence):
    return most_similar_w2v(sentence, feelings_word2vec_matrix)

def most_similar_by_lyrics_w2v(sentence):
    return most_similar_w2v(sentence, lyrics_word2vec_matrix)

#### Teste

In [13]:
test_similarity_function(
    most_similar_by_feeling_w2v,
    most_similar_by_lyrics_w2v
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected song (Station To Station) true position:
		* By feeling: 1º
		* By lyrics: 41º
	* Most similar songs by feeling: 
		1. Station To Station: 99.28%
		2. Aladdin Sane (1913-1938-197?): 69.42%
		3. 1917: 69.42%
	* Most similar songs by lyrics: 
		1. The Supermen: 71.94%
		2. Too Dizzy: 69.27%
		3. That's Motivation: 69.24%


Input phrase: I feel like I'm cracking under pressure and that only love can save me.
	* Expected song (Under Pressure) true position:
		* By feeling: 5º
		* By lyrics: 285º
	* Most similar songs by feeling: 
		1. Knock On Wood: 68.68%
		2. Don't Look Down: 66.13%
		3. Can You Hear Me: 64.43%
	* Most similar songs by lyrics: 
		1. Letter To Hermione: 75.47%
		2. Sweet Thing - (working lyrics): 73.49%
		3. If There Is Something: 73.01%


Input phrase: I feel euphoric and want to dance with my beloved.
	* Expected 

Essa abordagem apresenta resultados melhores para a maioria dos casos, todavia resultados ruins em Let's Dance e I Would Be Your Slave acabaram levando a média para baixo.

Todavia, podemos observar melhora nos casos em que as sentenças contém sinônimos e palavras em relação ao modelo TF-IDF, como no caso de China Girl e Ashes To Ashes.

### Word2Vec + TF-IDF

Utilizar o Word2Vec sozinho pode não ser suficiente para capturar a importância relativa das palavras na sentença. Para melhorar a vetorização, podemos combinar o Word2Vec com o TF-IDF, ponderando os vetores das palavras pelo seu peso TF-IDF. Dessa forma, palavras mais importantes na sentença terão maior influência no vetor final.

Como utilizaremos o TF-IDF para ponderar os vetores das palavras, não é necessário remover as stopwords, pois o TF-IDF já atribui um peso menor a palavras comuns.

#### Vetorização

A vetorização de sentença é feita calculando a média ponderada dos vetores das palavras que compõem a sentença, utilizando os pesos TF-IDF das palavras. Se uma palavra não estiver presente no vocabulário do modelo ou no vocabulário do TF-IDF, ela é ignorada. Se nenhuma palavra da sentença estiver presente no vocabulário, um vetor de zeros é retornado.

In [14]:
def weighted_avg_w2v_vec(sentence, w2v_model, tfidf):
    words = tokenize(sentence.lower(), False)
    vectors = []
    weights = []

    for word in words:
        if word in w2v_model and word in tfidf:
            vectors.append(w2v_model[word])
            weights.append(tfidf[word])

    if not vectors:
        return np.zeros(w2v_model.vector_size)

    return np.average(vectors, axis=0, weights=weights)

feeling_weighted_word2vec_matrix = [
    weighted_avg_w2v_vec(feeling, word2vec_model, feelings_tfidf_vectorizer.vocabulary_)
    for feeling in lyrics_clean['feelings']
]

lyrics_weighted_word2vec_matrix = [
    weighted_avg_w2v_vec(lyric, word2vec_model, lyrics_tfidf_vectorizer.vocabulary_)
    for lyric in lyrics_clean['lyrics']
]


#### Função de similaridade

Mais uma vez, a função de similaridade é implementada da mesma forma que a função de similaridade baseada em TF-IDF, mas utilizando a vetorização de sentença baseada em Word2Vec ponderada pelo TF-IDF.

In [15]:
def most_similar_weighted_w2v(sentence, matrix, vocabulary):
    phrase_vec = weighted_avg_w2v_vec(sentence, word2vec_model, vocabulary).reshape(1, -1)
    return similaridade(phrase_vec, matrix)


def most_similar_by_feeling_weighted_w2v(sentence):
    return most_similar_weighted_w2v(
        sentence, feeling_weighted_word2vec_matrix, feelings_tfidf_vectorizer.vocabulary_
    )


def most_similar_by_lyrics_weighted_w2v(sentence):
    return most_similar_weighted_w2v(
        sentence, lyrics_weighted_word2vec_matrix, lyrics_tfidf_vectorizer.vocabulary_
    )

In [16]:
test_similarity_function(
    most_similar_by_feeling_weighted_w2v,
    most_similar_by_lyrics_weighted_w2v
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected song (Station To Station) true position:
		* By feeling: 1º
		* By lyrics: 23º
	* Most similar songs by feeling: 
		1. Station To Station: 98.85%
		2. Aladdin Sane (1913-1938-197?): 68.70%
		3. The Width Of A Circle: 68.55%
	* Most similar songs by lyrics: 
		1. Soul Love: 66.63%
		2. God Knows I'm Good: 66.22%
		3. The Drowned Girl: 66.19%


Input phrase: I feel like I'm cracking under pressure and that only love can save me.
	* Expected song (Under Pressure) true position:
		* By feeling: 28º
		* By lyrics: 1º
	* Most similar songs by feeling: 
		1. God Only Knows: 75.37%
		2. Baby It Can't Fall: 72.31%
		3. Quicksand: 71.97%
	* Most similar songs by lyrics: 
		1. Under Pressure: 83.33%
		2. All The Madmen: 78.09%
		3. Let's Spend The Night Together: 77.87%


Input phrase: I feel euphoric and want to dance with my beloved.
	* E

O pior resultado até agora para similaridade por sentimento, todavia, o melhor resultado para similaridade por letra. Provavelmente pela alta repetição de palavras nas letras, como no caso de refrões, que acabam distorcendo a média dos vetores das palavras. A utilização do TF-IDF para ponderar os vetores das palavras acaba ajudando a reduzir o impacto dessas palavras comuns, melhorando a performance do modelo.

### Doc2Vec

Uma outra abordagem de vetorização é a utilização do modelo Doc2Vec, uma extensão do Word2Vec que permite gerar vetores para documentos inteiros, ao invés de apenas palavras. Dispensando assim a etapa de calculo da média vetorial.

In [17]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

def build_doc2vec(label: str):
    tags = [
        TaggedDocument(words=tokenize(s, False), tags=[str(i)])
        for i, s in enumerate(lyrics_clean[label])
    ]

    d2v = Doc2Vec(vector_size=100, window=5, min_count=1, workers=4, epochs=40)
    d2v.build_vocab(tags)
    d2v.train(tags, total_examples=d2v.corpus_count, epochs=d2v.epochs)

    return d2v

feelings_doc2vec = build_doc2vec("feelings")
lyrics_doc2vec = build_doc2vec("lyrics")

#### Função de similaridade

Aqui, a implementação da função difere ligeiramente, ao invés de utilizar a função `similaridade` anteriormente definido, utilizaremos o prório método `most_similar` do objeto de tipo `Doc2Vec`. Que utiliza, pela documentação ([most_similar](https://tedboy.github.io/nlps/generated/generated/gensim.models.Doc2Vec.most_similar.html)), uma implementação de similaridade de cosseno.

In [18]:
def most_similar_doc2vec(sentence, d2v_vectors):
    inferred_vector = d2v_vectors.infer_vector(word_tokenize(sentence.lower()))
    most_similar = d2v_vectors.dv.most_similar([inferred_vector], topn=len(d2v_vectors.dv))
    idxs = []
    sims = []
    for tag, sim in most_similar:
        idxs.append(int(tag))
        sims.append(sim)
    return idxs, sims

def most_similar_by_feeling_doc2vec(sentence):
    return most_similar_doc2vec(sentence, feelings_doc2vec)

def most_similar_by_lyrics_doc2vec(sentence):
    return most_similar_doc2vec(sentence, lyrics_doc2vec)

In [19]:
test_similarity_function(
    most_similar_by_feeling_doc2vec,
    most_similar_by_lyrics_doc2vec
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected song (Station To Station) true position:
		* By feeling: 1º
		* By lyrics: 155º
	* Most similar songs by feeling: 
		1. Station To Station: 99.61%
		2. Amsterdam: 99.50%
		3. Aladdin Sane (1913-1938-197?): 99.49%
	* Most similar songs by lyrics: 
		1. Neuköln: 49.58%
		2. Needles On The Beach: 48.04%
		3. Sense Of Doubt: 47.75%


Input phrase: I feel like I'm cracking under pressure and that only love can save me.
	* Expected song (Under Pressure) true position:
		* By feeling: 164º
		* By lyrics: 1º
	* Most similar songs by feeling: 
		1. Survive: 94.34%
		2. Conversation Piece: 94.14%
		3. And I Say To Myself: 94.11%
	* Most similar songs by lyrics: 
		1. Under Pressure: 49.85%
		2. I Feel Free: 41.78%
		3. Movin' On: 40.57%


Input phrase: I feel euphoric and want to dance with my beloved.
	* Expected song (Let's Dance) true p

Excetuando Under Pressure, os resultados dessa abordagem foram muito aquém, recomendando músicas irrelevantes para a maioria dos casos. Provavelmente devido ao tamanho do dataset, que é relativamente pequeno para o treinamento de um modelo de embedding.

## Transformers

O padrão ouro da vetorização de sentenças, transformers, como descritos no paper Attention is All You Need, levam em consideração a relação entre as palavras da sentença, permitindo assim vetores que levam em consideração contexto.

Utilizaremos a biblioteca `SentenceTransformer`, que já implementa todos os passos necessários, desde o download do modelo do `HuggingFace` bem como o processo de inferência.

In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("google/embeddinggemma-300m")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

### Vetorização

Aqui, simplesmente utilizamos o método `encode` do modelo.

In [21]:
feelings_transformer_matrix = model.encode(lyrics_clean['feelings'].to_list())
lyrics_transformer_matrix = model.encode(lyrics_clean['lyrics'].to_list())

### Função de Similaridade

Vetorizamos a sentença e calculamos a similardade com a matriz de sentimento/letras, da mesma maneira que fizemos anteriormente, a própria biblioteca já se carrega do processo de tokenização e pré-processamento.

In [22]:
def most_similar_transformer(sentence, matrix):
    phrase_vec = model.encode([sentence])
    return similaridade(phrase_vec, matrix)

def most_similar_by_feeling_transformer(sentence):
    return most_similar_transformer(sentence, feelings_transformer_matrix)

def most_similar_by_lyrics_transformer(sentence):
    return most_similar_transformer(sentence, lyrics_transformer_matrix)


In [23]:
test_similarity_function(
    most_similar_by_feeling_transformer,
    most_similar_by_lyrics_transformer
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected song (Station To Station) true position:
		* By feeling: 1º
		* By lyrics: 91º
	* Most similar songs by feeling: 
		1. Station To Station: 85.73%
		2. China Girl: 59.10%
		3. Turn Blue: 58.62%
	* Most similar songs by lyrics: 
		1. The Voyeur Of Utter Destruction (As Beauty): 40.60%
		2. Ballad Of The Adventurers: 40.23%
		3. Unwashed and Somewhat Slightly Dazed: 40.06%


Input phrase: I feel like I'm cracking under pressure and that only love can save me.
	* Expected song (Under Pressure) true position:
		* By feeling: 18º
		* By lyrics: 1º
	* Most similar songs by feeling: 
		1. Amazing: 57.62%
		2. A Better Future: 53.76%
		3. Don't Let Me Down And Down: 52.91%
	* Most similar songs by lyrics: 
		1. Under Pressure: 55.15%
		2. Run: 46.86%
		3. I Can't Explain: 46.85%


Input phrase: I feel euphoric and want to dance with my be

O resultado médio é o melhor até agora, e embora não tenha colocado a música esperada no top 3 com a mesma frequência que as outras abordagens. Todavia, a música recomendada é muito mais condizente com o sentimento analisado.

# Outras Abordagens

A abordagem de semelhança de cosseno utilizando vetorização de sentenças é uma forma eficaz de recomendar músicas com base no sentimento do usuário. Todavia, uma abordagem alternativa seria possível com classficação com os seguintes passos:

1. Anotar as músicas com múltiplos rótulos de sentimentos.
2. Realizar o treinamento de um modelo de classificação multi-rótulo.
3. Utilizar o modelo para classificar a entrada do usuário.
4. Recomendar músicas cujo o conjunto de rótulos seja mais semelhante ao conjunto da entrada.

Essa abordagem foi preterida para esse projeto por exigir um trabalho muito grande para a anotação das músicas e pela natureza "lossy" da classificação multi-rótulo, que poderia levar a perda de informações importantes sobre o sentimento da música.

# Implementando o sistema

In [24]:
class Model:
    TFIDF = "TF-IDF"
    WORD2VEC = "Word2Vec"
    WEIGHTED_WORD2VEC = "Weighted Word2Vec"
    DOC2VEC = "Doc2Vec"
    TRANSFORMER = "Transformer"

    def __init__(self, most_similar_by_feeling_function, most_similar_by_lyrics_function, name):
        self.name = name
        self.most_similar_by_feeling = most_similar_by_feeling_function
        self.most_similar_by_lyrics = most_similar_by_lyrics_function

models = {
    "I": Model(
        most_similar_by_feeling_tfidf,
        most_similar_by_lyrics_tfidf,
        Model.TFIDF
    ),
    "V": Model(
        most_similar_by_feeling_w2v,
        most_similar_by_lyrics_w2v,
        Model.WORD2VEC
    ),
    "W": Model(
        most_similar_by_feeling_weighted_w2v,
        most_similar_by_lyrics_weighted_w2v,
        Model.WEIGHTED_WORD2VEC
    ),
    "D": Model(
        most_similar_by_feeling_doc2vec,
        most_similar_by_lyrics_doc2vec,
        Model.DOC2VEC
    ),
    "T": Model(
        most_similar_by_feeling_transformer,
        most_similar_by_lyrics_transformer,
        Model.TRANSFORMER
    ),
}

In [27]:
from IPython.core.display_functions import clear_output

def on_all_models(sent):
    return {
        k: (
            m.most_similar_by_feeling(sent),
            m.most_similar_by_lyrics(sent)
        )
        for k, m in models.items()
    }

def run():
    print("Please enter a sentence describing your current mood or feeling:")
    user_input = input("> ")
    clear_output(wait=True)

    print(f'\nI think I know just the right song for "{user_input}"\n')

    results = on_all_models(user_input)

    for k, ((feeling_indices, feeling_similarities), (lyrics_indices, lyrics_similarities)) in results.items():
        model_name = models[k].name
        print(f"Model: {model_name}")
        print("Top 3 songs based on feelings (Similarity):")
        for i in range(3):
            song_idx = feeling_indices[i]
            sim_score = feeling_similarities[i]
            print(f"{i + 1}. {lyrics_clean.iloc[song_idx]['title']} ({sim_score:.2%})")

        print("\nTop 3 songs based on lyrics (Similarity):")
        for i in range(3):
            song_idx = lyrics_indices[i]
            sim_score = lyrics_similarities[i]
            print(f"{i + 1}. {lyrics_clean.iloc[song_idx]['title']} ({sim_score:.2%})")
        print("\n" + "-"*50 + "\n")

In [28]:
run()


I think I know just the right song for "I feel scared, as if I've met my own doppleganger"

Model: TF-IDF
Top 3 songs based on feelings (Similarity):
1. Uncle Floyd (18.83%)
2. Underground (17.20%)
3. The Man Who Sold The World (14.73%)

Top 3 songs based on lyrics (Similarity):
1. Afraid (18.87%)
2. Can't Help Thinking About Me (14.16%)
3. The London Boys (13.68%)

--------------------------------------------------

Model: Word2Vec
Top 3 songs based on feelings (Similarity):
1. C'est La Vie (63.31%)
2. What Kind Of Fool Am I? (62.46%)
3. Nature Boy (Version 1) (62.04%)

Top 3 songs based on lyrics (Similarity):
1. Afraid (71.17%)
2. Don't Be Afraid (alias Oh Darling) (69.21%)
3. Girls (69.11%)

--------------------------------------------------

Model: Weighted Word2Vec
Top 3 songs based on feelings (Similarity):
1. Standing Next To You (65.21%)
2. What Kind Of Fool Am I? (65.09%)
3. Nature Boy (Version 1) (64.15%)

Top 3 songs based on lyrics (Similarity):
1. Can't Help Thinking Abo